In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import f1_score, accuracy_score

In [3]:
TRAIN_CSV = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/train/train.csv"
# TEST_CSV = "/kaggle/input/datasets/keshavvenkateshirv/tamil-misogyny-meme-detection/test/test/test.csv"
TEST_CSV = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/test/test.csv"
DEV_CSV = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/dev/dev.csv"

import pandas as pd

train_df = pd.read_csv(TRAIN_CSV, encoding="utf-8")
dev_df   = pd.read_csv(DEV_CSV, encoding="utf-8")
test_df  = pd.read_csv(TEST_CSV, encoding="utf-8")
# test_lab_df = pd.read_csv(TEST_LAB_CSV, encoding="utf-8")

train_df["image_id"] = train_df["image_id"].astype(str)
dev_df["image_id"]   = dev_df["image_id"].astype(str)
test_df["image_id"]  = test_df["image_id"].astype(str)

train_image_dir = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/train"
dev_image_dir   = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/dev"
test_image_dir  = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/test"

In [4]:
train_df.head()

,image_id,transcriptions,original_labels,irish_labels,chinese_labels
0,654,She: Recharge panni vidu da ~En unta kaasu ill...,not-misogyny,not-misogyny,not-misogyny
1,1339,That look from wife wen u ask her BJ after 3 r...,misogyny,misogyny,misogyny
2,288,* 10mins into the online class* My mind : innu...,not-misogyny,not-misogyny,not-misogyny
3,1294,avar munnadi thangachi thangachi nu paasama ko...,misogyny,misogyny,misogyny
4,798,LakshmiMV @mvlakshmibe56 Stopped exercising an...,not-misogyny,not-misogyny,not-misogyny


In [ ]:
dev_df.head()

In [6]:
test_df.head()

,image_id,transcriptions,original_labels,irish_labels,chinese_labels
0,1193,கணவர் : செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கட...,NaN,NaN,NaN
1,427,MUTHU MC @muthu__kutti *Manager discussing wit...,NaN,NaN,NaN
2,1474,Viper Memez IRUKKU EDHO ULLA IRUKKU,NaN,NaN,NaN
3,766,In Irugapatru Director Yuvaraj Dhayalan has gi...,NaN,NaN,NaN
4,486,Dad : யாரையாவது Love பண்ணா சொல்றா கட்டிவைக்கிற...,NaN,NaN,NaN


In [ ]:
# MODEL_NAME = "l3cube-pune/tamil-bert"
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_encoder = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
'''
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

from transformers import ViTImageProcessor

image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
'''

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

import torchvision.models as models
import torch.nn as nn

image_encoder = models.resnet50(pretrained=True)
# self.image_encoder.fc = nn.Identity()   # remove classifier

In [ ]:
class TamilMemeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, transform, image_dir, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.transform = transform
        self.image_dir = image_dir
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT
        text = str(row["transcriptions"])

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # IMAGE
        image_filename = str(row["image_id"])

        # Add .jpg if missing
        if not image_filename.endswith(".jpg"):
            image_filename = image_filename + ".jpg"

        image_path = os.path.join(self.image_dir, image_filename)

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        # LABEL
        label_value = row["original_labels"]

        # If it's string label like "misogynous"
        if isinstance(label_value, str):
            label_value = label_value.strip().lower()
            
            # map labels
            label_map = {
                "not-misogyny": 0,
                "misogyny": 1
            }
            
            label_value = label_map.get(label_value, 0)
        
        # convert to float tensor
        label = torch.tensor(label_value, dtype=torch.float)
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image": image,
            "original_labels": label   # ❗ NOT "original_labels"
        }

In [ ]:
train_dataset = TamilMemeDataset(train_df, tokenizer, image_transform, train_image_dir)
dev_dataset   = TamilMemeDataset(dev_df, tokenizer, image_transform, dev_image_dir)
test_dataset  = TamilMemeDataset(test_df, tokenizer, image_transform, test_image_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)



In [ ]:
class BaselineMultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TEXT
        self.text_encoder = AutoModel.from_pretrained(MODEL_NAME)
        text_hidden_size = 768

        # IMAGE
        self.image_encoder = models.resnet50(pretrained=True)
        image_hidden_size = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()

        # CLASSIFIER
        self.classifier = nn.Sequential(
            nn.Linear(text_hidden_size + image_hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, input_ids, attention_mask, images):

        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_embedding = text_outputs.pooler_output

        image_embedding = self.image_encoder(images)

        combined = torch.cat((text_embedding, image_embedding), dim=1)

        logits = self.classifier(combined)

        return logits.squeeze(1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineMultimodalModel().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)
        labels = batch["original_labels"].to(device).float()

        if labels.dim() == 1:
            labels = labels.unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask, images)

        # FIX SHAPE HERE
        if outputs.dim() == 1:
            outputs = outputs.unsqueeze(1)
        
        labels = labels.float()
        
        
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader, threshold=0.5):
    model.eval()

    all_preds = []
    all_labels = []
    has_labels = True

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["image"].to(device)

            outputs = model(input_ids, attention_mask, images)
            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).float()

            all_preds.extend(preds.cpu().numpy())

            # Check if labels exist
            if "original_labels" in batch:
                labels = batch["original_labels"].to(device)
                if (labels != -1).all():
                    all_labels.extend(labels.cpu().numpy())
                else:
                    has_labels = False
            else:
                has_labels = False

    if has_labels and len(all_labels) > 0:
        macro_f1 = f1_score(all_labels, all_preds, average="macro")
        accuracy = accuracy_score(all_labels, all_preds)
        return macro_f1, accuracy
    else:
        return all_preds  # return predictions only

In [ ]:
EPOCHS = 10

best_dev_f1 = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    dev_f1, dev_acc = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Dev Macro-F1: {dev_f1:.4f}")
    print(f"Dev Accuracy: {dev_acc:.4f}")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        torch.save(model.state_dict(), "best_baseline_tamil.pt")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def get_probs_and_labels(model, loader):
    model.eval()
    
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["image"].to(device)

            # label handling (robust)
            if "labels" in batch:
                labels = batch["labels"].to(device)
            elif "original_labels" in batch:
                labels = batch["original_labels"].to(device)
            else:
                labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask, images)
            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    return np.array(all_probs), np.array(all_labels)


In [ ]:
def find_best_threshold(probs, labels):
    best_thresh = 0.5
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.02):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print("Best Threshold:", best_thresh)
    print("Best F1:", best_f1)

    return best_thresh

In [ ]:
probs, labels = get_probs_and_labels(model, dev_loader)

best_threshold = find_best_threshold(probs, labels)

In [ ]:
model.load_state_dict(torch.load("best_baseline_tamil.pt"))

dev_f1, dev_acc = evaluate(model, dev_loader)
print("Dev Macro-F1:", dev_f1)
print("Dev Accuracy:", dev_acc)

In [ ]:
test_df = pd.read_csv(TEST_CSV)

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

def predict(model, loader, threshold=0.5):
    model.eval()

    all_ids = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["image"].to(device)

            logits = model(input_ids, attention_mask, images)
            probs = torch.sigmoid(logits)

            preds = (probs >= threshold).int()

            # flatten
            preds = preds.cpu().numpy().flatten()

            all_preds.extend(preds)

            # collect ids
            if "id" in batch:
                all_ids.extend(batch["id"])
    
    return all_ids, all_preds



In [ ]:
test_df = pd.read_csv(TEST_CSV)

ids, preds = predict(model, test_loader, threshold=best_threshold)

test_df["original_labels"] = preds

test_df.to_csv("test_with_predictions.csv", index=False)

In [ ]:
test_df.to_csv("test_with_predictions.csv", index=False)

In [ ]:
pred_df = pd.read_csv("/kaggle/working/test_with_predictions.csv")

In [ ]:
pred_df.columns

In [ ]:
pred_df['original_labels'].info()

In [ ]:
pred_df.head()

In [5]:
# =========================
# 🔧 INSTALLS (if needed)
# =========================
#!pip install -q transformers timm scikit-learn

# =========================
# 📦 IMPORTS
# =========================
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import f1_score
from tqdm import tqdm

# =========================
# 📂 PATHS
# =========================
TRAIN_CSV = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/train/train.csv"
DEV_CSV   = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/dev/dev.csv"
TEST_CSV  = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/test/test.csv"

train_image_dir = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/train"
dev_image_dir   = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/dev"
test_image_dir  = "/kaggle/input/datasets/keshavvenkateshirv/codabench-misogyny/test"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 📊 LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
test_df  = pd.read_csv(TEST_CSV)

# ✅ FIXED BINARY LABEL MAPPING
label2id = {"not-misogyny": 0, "misogyny": 1}
id2label = {0: "not-misogyny", 1: "misogyny"}
train_df['label'] = train_df['original_labels'].map(label2id)
dev_df['label']   = dev_df['original_labels'].map(label2id)

# ✅ FIX: remove rows where mapping failed
train_df = train_df.dropna(subset=['label']).reset_index(drop=True)
dev_df   = dev_df.dropna(subset=['label']).reset_index(drop=True)

# Convert to int explicitly
train_df['label'] = train_df['label'].astype(int)
dev_df['label']   = dev_df['label'].astype(int)
NUM_CLASSES = 2

# =========================
# 🔤 TOKENIZER (MuRIL)
# =========================
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# =========================
# 🖼️ IMAGE TRANSFORMS
# =========================
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# =========================
# 🧠 DATASET
# =========================
class MultiModalDataset(Dataset):
    def __init__(self, df, image_dir, tokenizer, is_test=False):
        self.df = df
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row['transcriptions'])
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        img_path = os.path.join(self.image_dir, str(row['image_id']))
        if not os.path.exists(img_path):
            img_path += ".jpg"

        image = Image.open(img_path).convert("RGB")
        image = transform(image)

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row['label'], dtype=torch.long)

        return item

# =========================
# 🧠 MODEL
# =========================
class MultiModalModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        self.image_model = models.resnet18(pretrained=True)
        self.image_model.fc = nn.Identity()

        self.fc = nn.Sequential(
            nn.Linear(768 + 512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, image):
        text_out = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        img_out = self.image_model(image)

        combined = torch.cat([text_out, img_out], dim=1)
        return self.fc(combined)

# =========================
# 🏋️ TRAIN SETUP
# =========================
train_dataset = MultiModalDataset(train_df, train_image_dir, tokenizer)
dev_dataset   = MultiModalDataset(dev_df, dev_image_dir, tokenizer)
test_dataset  = MultiModalDataset(test_df, test_image_dir, tokenizer, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

model = MultiModalModel(NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# =========================
# 🏋️ TRAIN LOOP
# =========================
EPOCHS = 10   # ✅ CHANGED

best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        images = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_ids, attention_mask, images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, images)
            probs = torch.softmax(logits, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_thr = 0.5
    best_epoch_f1 = 0

    for thr in np.linspace(0.2, 0.8, 20):
        preds = (all_probs == all_probs.max(axis=1, keepdims=True)).astype(int)
        preds = preds.argmax(axis=1)

        f1 = f1_score(all_labels, preds, average='macro')

        if f1 > best_epoch_f1:
            best_epoch_f1 = f1
            best_thr = thr

    print(f"Dev F1: {best_epoch_f1:.4f} | Best Thr: {best_thr}")

    if best_epoch_f1 > best_f1:
        best_f1 = best_epoch_f1
        torch.save(model.state_dict(), "best_model.pt")

# =========================
# 🧪 TEST PREDICTION
# =========================
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        images = batch['image'].to(DEVICE)

        logits = model(input_ids, attention_mask, images)
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())

test_df['original_labels'] = [id2label[p] for p in predictions]
''
# =========================
# 💾 SAVE SUBMISSION
# =========================
test_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Epoch 1 Loss: 0.5223901561448272
Dev F1: 0.5478 | Best Thr: 0.2


100%|██████████| 71/71 [00:47<00:00,  1.48it/s]


Epoch 2 Loss: 0.3990841505812927
Dev F1: 0.7075 | Best Thr: 0.2


...

Epoch 8 Loss: 0.01946676453627961
Dev F1: 0.7731 | Best Thr: 0.2


100%|██████████| 71/71 [00:47<00:00,  1.50it/s]


Epoch 9 Loss: 0.018273541953047395
Dev F1: 0.7571 | Best Thr: 0.2


100%|██████████| 71/71 [00:47<00:00,  1.51it/s]


Epoch 10 Loss: 0.010996103614673649
Dev F1: 0.7725 | Best Thr: 0.2
Saved submission.csv


In [6]:
label2id = {"not-misogyny": 0, "misogyny": 1}
test_df['original_labels'] = test_df['original_labels'].map(label2id)

In [7]:
test_df.head()

,image_id,transcriptions,original_labels,irish_labels,chinese_labels
0,1193,கணவர் : செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கட...,1,NaN,NaN
1,427,MUTHU MC @muthu__kutti *Manager discussing wit...,0,NaN,NaN
2,1474,Viper Memez IRUKKU EDHO ULLA IRUKKU,0,NaN,NaN
3,766,In Irugapatru Director Yuvaraj Dhayalan has gi...,0,NaN,NaN
4,486,Dad : யாரையாவது Love பண்ணா சொல்றா கட்டிவைக்கிற...,0,NaN,NaN


In [9]:
test_df = test_df.drop(columns=['transcriptions', 'irish_labels', 'chinese_labels'], errors='ignore')

In [10]:
test_df.head()

,image_id,original_labels
0,1193,1
1,427,0
2,1474,0
3,766,0
4,486,0


In [11]:
test_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
